# Computer lab 2: convex and nonlinear optimization

**Mathematical Optimization, VU Amsterdam, 2026–27.**

For each problem, identify the domain and justify your convexity claim before
interpreting a numerical solution. Plots and repeated convergence from different
starting points can guide exploration, but neither proves convexity or global
optimality.

## Setup

Run the next two cells before starting. Colab installs the required packages and
**Ipopt**, the nonlinear solver used here. The setup code can be expanded for
inspection. If asked to restart the session, restart and run Setup again.
See the [course setup instructions](https://github.com/alessandrozocca/MO2026#computer-lab-setup)
for local use and troubleshooting.

In [ ]:
#@title Prepare course helpers and solvers
import hashlib
import os
from pathlib import Path
import sys
import types
from urllib.request import urlopen

SETUP_COMMIT = '8b93f07ad4680561374d6c7a9645a2d89568bf43'
SETUP_SHA256 = "92575ba111b65fdca951bd0cab948c64a47c2d3f75c201e8194bef87cc97e938"
local_repo = os.environ.get("MO2026_REPO")
if local_repo:
    setup_location = str(Path(local_repo).expanduser() / "lab_setup.py")
    setup_source = Path(setup_location).read_bytes()
else:
    if SETUP_COMMIT is None:
        raise RuntimeError("This draft needs its published course setup pin before use on Colab.")
    setup_location = f"https://raw.githubusercontent.com/alessandrozocca/MO2026/{SETUP_COMMIT}/lab_setup.py"
    with urlopen(setup_location, timeout=30) as response:
        setup_source = response.read()
    if hashlib.sha256(setup_source).hexdigest() != SETUP_SHA256:
        raise RuntimeError("Course setup checksum mismatch; the downloaded code was not executed.")

lab_setup = types.ModuleType("mo2026_lab_setup")
lab_setup.__file__ = setup_location
sys.modules[lab_setup.__name__] = lab_setup
exec(compile(setup_source, setup_location, "exec"), lab_setup.__dict__)
if lab_setup.API_VERSION != 1:
    raise RuntimeError("This notebook requires course setup API 1.")
lab = lab_setup.setup(week=2)

In [ ]:
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyomo.environ as pyo
import sympy as sp
from IPython.display import display

nlp_solver = lab.nlp_solver
plotting = lab.helpers["tutorial2"]
plot_feasible_region = plotting.plot_feasible_region
plot_objective = plotting.plot_objective
format_number = lab_setup.format_number
x1_sym, x2_sym = sp.symbols("x_1 x_2", real=True)

Ipopt checks local optimality conditions. Its `optimal` label does **not** certify
global optimality for a nonconvex problem; Exercise 3 has different local minima.
Global conclusions below use convexity and independent mathematical checks.
The function below checks termination before loading a solution. Pass the solver
explicitly, then evaluate with `pyo.value(...)`. `format_number` changes text only.

In [ ]:
def solve_and_load(model, solver):
    result = solver.solve(model, load_solutions=False)
    pyo.assert_optimal_termination(result)
    model.solutions.load_from(result)
    return result

Useful references are [Chapter 5: Convex Optimization](https://mobook.github.io/MO-book/notebooks/05/05.00.html),
[Chapter 6: Conic Optimization](https://mobook.github.io/MO-book/notebooks/06/06.00.html)
and the [Ipopt documentation](https://coin-or.github.io/Ipopt/).
The finite windows in the plots are viewing windows, not additional constraints.

## Exercise 1: checking convexity

Consider

$$
\begin{aligned}
\min\quad &x_1^2+x_2\\
\text{s.t.}\quad &-x_1+x_2^2\le1,\\
&x_1+x_2\ge4,\qquad x_1,x_2\in\mathbb{R}.
\end{aligned}
$$

Is this a convex optimization problem? Justify your answer using the objective
and the constraint functions. Implement and solve it, then report the decisions
and objective value.

In [ ]:
plot_1 = plot_feasible_region(
    [-x1_sym + x2_sym**2 <= 1, x1_sym + x2_sym >= 4],
    (x1_sym, 0, 6), (x2_sym, -3, 3), show=False, size=(7, 5),
)
plot_objective(plot_1, x1_sym**2 + x2_sym, [7, 10, 15, 20, 30])

In [ ]:
# Explain convexity using Hessians or known convex functions.
# Build the model with real variables and solve using solve_and_load(model, nlp_solver).
# Report the decisions and objective value; add its level curve to the plot if useful.

## Exercise 2: exponential and quartic functions

Consider

$$
\begin{aligned}
\min\quad &x_1+e^{x_1+x_2}\\
\text{s.t.}\quad &x_1^4+x_2^4\le1,\\
&x_1-e^{x_2}\ge0,\qquad x_1,x_2\in\mathbb{R}.
\end{aligned}
$$

Justify whether this is a convex optimization problem. Put the inequalities in
the appropriate orientation for that argument, implement the model and solve it.
Use `pyo.exp` in Pyomo expressions and `sp.exp` for symbolic plots.

In [ ]:
plot_2 = plot_feasible_region(
    [x1_sym**4 + x2_sym**4 <= 1, sp.exp(x2_sym) - x1_sym <= 0],
    (x1_sym, 0, 1.05), (x2_sym, -1.05, 0.05), show=False, size=(7, 5),
)
plot_objective(plot_2, x1_sym + sp.exp(x1_sym + x2_sym), [0.8, 1.2, 1.6, 2.2, 3])

In [ ]:
# Write both inequalities as convex functions <= 0.
# Explain the convexity of the quartic and exponential terms.
# Implement, solve and report the solution using Ipopt.

## Exercise 3: local minima and a convex restriction

Consider

$$
\min_{-3.5\le x\le3.5} f(x),\qquad f(x)=x^4-11x^2-\frac{x}{10}.
$$

1. Compute $f''(x)$ and decide whether the problem is convex.
2. Solve it with Ipopt starting at $x=-2$ and $x=2$. Compare the returned points
   and function values. What can the solver's termination label establish here?
3. Does adding $x\ge0$ make the problem convex? Use $f''$ to decide, even if
   different starts converge to the same point.
4. Find a lower bound $a>0$ such that the restriction $a\le x\le3.5$ is convex.
   Solve that restriction from several valid starting points. Explain why it
   retains the original global minimizer for this particular function.

Restricting a domain generally changes the optimization problem; retaining a
global minimizer needs a separate argument.

In [ ]:
def quartic(value):
    return value**4 - 11 * value**2 - value / 10

quartic_grid = np.linspace(-3.5, 3.5, 600)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
axes[0].plot(quartic_grid, quartic(quartic_grid))
axes[0].set(xlabel="x", ylabel="f(x)", title="Objective on the stated interval")
axes[1].plot(quartic_grid, 12 * quartic_grid**2 - 22)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set(xlabel="x", ylabel="f''(x)", title="Second derivative")
plt.show()

In [ ]:
# Build a function returning an unsolved model with bounds (-3.5, 3.5).
# Solve from -2 and 2, then compare with the nonnegative and convex restrictions.
# Choose starts within each restricted interval and report points and objective values.
# Distinguish the local termination check from a global conclusion.

In [ ]:
# Find the real roots of f'(x), keep those inside [-3.5, 3.5], and include both endpoints.
# Compare their objective values to check the global minimum independently.

## Exercise 4: domains and an epigraph

Consider

$$
\begin{aligned}
\min\quad &\max\left\{\frac1{x_1-x_2},\frac{(x_1+4x_2)^2}{x_2}\right\}\\
\text{s.t.}\quad &x_1+x_2\ge\frac23,\qquad x_1,x_2\ge0.
\end{aligned}
$$

The objective is defined only when $x_2>0$ and $x_1\ne x_2$.

1. Explain why the natural feasible domain is not convex. Show that restricting
   to $x_1>x_2>0$ gives a convex problem.
2. Introduce an epigraph variable for the maximum and implement the positive-domain
   problem. Keep every denominator positive during the numerical solve.
3. Report the solution and check both original objective terms. Does the restricted
   solution also minimize the objective over the full natural domain? Justify
   that conclusion mathematically.

In [ ]:
# Minimize an epigraph variable T with two inequalities.
# Reparametrize with a positive gap u=x1-x2 and positive v=x2.
# Explain any numerical lower bounds, initialize inside them, and use Ipopt.
# Convert back to x1,x2 and check the two original objective terms.

In [ ]:
# Optional: plot objective contours only where x1>x2>0 and x1+x2>=2/3.
# Mask invalid points before evaluating either denominator.

## Exercise 5: ordinary least squares and a conic formulation

For observations $i=0,\ldots,n-1$ with features $X_{ij}$, $j=0,\ldots,k-1$, fit
an intercept $b$ and slopes $m_j$ by ordinary least squares (OLS):

$$
\begin{aligned}
\min\quad &\sum_{i=0}^{n-1}e_i^2\\
\text{s.t.}\quad &e_i=y_i-b-\sum_{j=0}^{k-1}X_{ij}m_j\quad\text{for each }i,\\
&b,m_j,e_i\in\mathbb{R}.
\end{aligned}
$$

1. Implement a function `build_ols(X, y)` returning an unsolved model. Fit the
   supplied data with Ipopt and report the intercept, slopes and sum of squared
   residuals (SSE).
2. Prove that the problem is convex. When are the fitted coefficients unique?
3. Introduce $t$ and write a second-order cone formulation minimizing the
   **Euclidean residual norm**. Explain why it gives the same fitted coefficients
   as OLS, and report its optimal value using your fit. Distinguish that value
   from the SSE.
4. Compare your coefficients and residuals with `np.linalg.lstsq`, including a
   column of ones for the intercept.

Use the smooth squared objective for the numerical Ipopt solve. The conic part
asks for the formulation and its relation to that solution.

In [ ]:
from sklearn.datasets import make_regression

X, y = make_regression(n_samples=500, n_features=5, noise=75, random_state=2022)
print(f"Data: {X.shape[0]} observations, {X.shape[1]} features; response shape {y.shape}")

In [ ]:
# Build a model with an intercept, k slopes and n residual variables.
# Add the affine residual equalities and minimize the sum of squared residuals.
# Solve with Ipopt, extract full-precision coefficients, and recompute residuals.
# Explain convexity and the full-column-rank condition for coefficient uniqueness.

In [ ]:
# Write the norm epigraph/SOC formulation and explain its relation to squared error.
# Form A with an intercept column, call np.linalg.lstsq, and compare the fits.

## Exercise 6: Markowitz portfolio optimization

An investor allocates capital $C$ among $n$ risky assets and a risk-free asset.
Let $x_i$ be the amount invested in risky asset $i$, and $\tilde x$ the risk-free
amount. Short selling and borrowing are excluded.

The risky **gross return factors** form a random vector $r$ with mean $\mu$ and
covariance $\Sigma$. A gross factor of 1.2 means that one invested unit becomes
1.2 units. The risk-free gross factor is $R$. Terminal wealth is
$W=R\tilde x+r^\top x$, so

$$
\mathbb{E}W=R\tilde x+\mu^\top x,\qquad
\operatorname{Var}(W)=x^\top\Sigma x.
$$

Maximize expected wealth subject to a variance limit $\gamma$:

$$
\begin{aligned}
\max\quad &R\tilde x+\sum_i\mu_i x_i\\
\text{s.t.}\quad &\sum_i x_i+\tilde x=C,\\
&x^\top\Sigma x\le\gamma,\\
&x_i\ge0,\quad\tilde x\ge0.
\end{aligned}
$$

1. Explain why the problem is convex when $\Sigma\succeq0$.
2. Implement a function `build_portfolio(data)` returning an unsolved model and
   solve the supplied instance. Infer the number of assets from the data.
3. Report the allocation, expected terminal wealth, expected gain above initial
   capital, and the variance and standard deviation of wealth. Check the budget
   and explain whether the risk limit is active.

These are **illustrative data**, with large return variances. Capital and wealth
are in currency units; $\gamma$ is in squared currency units. The standard
deviation is the square root of the variance, not the variance itself.

In [ ]:
@dataclass
class PortfolioData:
    capital: float
    variance_limit: float
    risk_free: float
    mean: np.ndarray
    covariance: np.ndarray

portfolio_data = PortfolioData(
    capital=1,
    variance_limit=1,
    risk_free=1.01,
    mean=np.array([1.2, 1.1, 1.3]),
    covariance=np.array([[1.5, 0.5, 2], [0.5, 2, 0], [2, 0, 5]]),
)
display(pd.DataFrame(
    {"Mean gross factor": portfolio_data.mean,
     "Gross-factor standard deviation": np.sqrt(np.diag(portfolio_data.covariance))},
    index=[f"Asset {i + 1}" for i in range(len(portfolio_data.mean))],
).style.format(format_number))

In [ ]:
# Explain convexity using the covariance matrix and affine objective/budget.
# Build nonnegative risky investments and a nonnegative risk-free amount.
# Write the expected wealth and variance as explicit indexed sums; solve with Ipopt.
# Recompute wealth, gain, variance, standard deviation and total investment from the data.